# Transcribing speech at a fraction of frontier costs

We've shown before that when it comes to speech transcription,
open models are [100x faster and 100x cheaper](https://modal.com/blog/fast-cheap-batch-transcription)
than proprietary APIs, and open models still occupy
[the top spots](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard).
But there's no reason to stop there: we can achieve state-of-the-art
performance by post-training open models to get even lower word error rates (WER).
As an example, we show how to post-train
[Qwen3-ASR-1.7B](https://huggingface.co/Qwen/Qwen3-ASR-1.7B) on the 
[hf-internal-testing/librispeech_asr_dummy](https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy)
dataset.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main
if importlib.util.find_spec('librosa') is None:
    %uv pip install -q soundfile librosa jiwer datasets

In [ ]:
from modal_training_gym import (
    CustomDeployment,
    MultimodalDataset,
    Qwen3_ASR_1_7B,
    Qwen3_ASR_1_7b_Recipe,
    TrainConfig,
    list_checkpoints,
)

## Deploy the base model

Since audio models are not yet supported on
[Endpoints](https://modal.com/docs/guide/endpoints), we use a
[CustomDeployment](https://gym.modal.dev/reference/deployment/customdeployment/)
to deploy the base and trained models.

In [ ]:
base_model = Qwen3_ASR_1_7B()
base_deployment = CustomDeployment.launch(
    base_model,
    unauthenticated=True,
    recreate_if_existing=True
)
base_deployment.wait_until_ready(timeout=15 * 60)
print(f"base model deployed to {base_deployment.url}")

## Define a scoring function

As mentioned before, we measure capability by lower WER, so that's what we'll use.
We can use the `jiwer` library to calculate this so we don't have to ourselves.

In [ ]:
async def score_transcript(response: str, label: str) -> float:
    import jiwer

    if not label:
        return 0.0
    return -float(jiwer.wer(label, response))

## Get the dataset

Since this dataset contains audio files, we create a `MultimodalDataset`
to pass the audio clips to rollouts. We do some pre-processing with
`soundfile` and store as base64 inline for demonstration purposes.
In a production use case, you'd likely instead store by reference.

In [ ]:
class LibriSpeechASRDataset(MultimodalDataset):
    modality = "audio"
    hf_repo = "hf-internal-testing/librispeech_asr_dummy"
    hf_config = "clean"
    hf_split = "validation"
    always_prepare = True
    apply_chat_template = False  # ensures the data URI is valid throughout the rollout

    def __init__(self, **kwargs):
        super().__init__(rows=[], **kwargs)

    def _build_rows(self) -> list[dict]:
        import base64 as b64
        import io

        import soundfile as sf
        from datasets import Audio, load_dataset

        ds = load_dataset(self.hf_repo, self.hf_config, split=self.hf_split)
        ds = ds.cast_column("audio", Audio(decode=False))  # decode with soundfile instead of torchcodec
        rows = []
        for ex in ds:
            audio = ex["audio"]
            data = (
                audio["bytes"]
                if audio.get("bytes")
                else open(audio["path"], "rb").read()
            )
            arr, sr = sf.read(io.BytesIO(data))
            buf = io.BytesIO()
            sf.write(buf, arr, sr, format="WAV")
            data_uri = "data:audio/wav;base64," + b64.b64encode(
                buf.getvalue()
            ).decode("ascii")
            rows.append(
                {
                    self.input_key: "<audio>\nTranscribe the speech to text. Respond with only the transcript.",
                    self.media_column: [data_uri],
                    self.label_key: ex["text"].lower().strip(),
                }
            )
        return rows

    def load(self, split: str = "all") -> list[dict]:
        return self._build_rows()

    def prepare(self, path, eval_paths=None):
        rows = self._build_rows()
        self._write_jsonl(rows, path)
        if eval_paths:
            for eval_path in eval_paths.values():
                self._write_jsonl(rows, eval_path)

train_dataset = LibriSpeechASRDataset(hf_split="validation[:8]")
eval_dataset = LibriSpeechASRDataset(hf_split="validation[8:16]")

Let's look at a row — text prompt, an audio data-URI, and the reference label.

In [ ]:
row = eval_dataset.load()[0]
print("prompt:", row["prompt"])
print("audio: ", row["audios"][0][:48], "...")
print("label: ", row["label"])

In [ ]:
async def word_error_rate_reward(args, sample, **kwargs) -> float:
    import jiwer

    response = (getattr(sample, "response", "") or "").lower().strip()
    reference = (getattr(sample, "label", "") or "").lower().strip()
    if not reference:
        return 0.0
    return -float(jiwer.wer(reference, response))

## Train

`Qwen3_ASR_1_7b_Recipe` carries the ASR-specific defaults — the transcription
rollout, padded (bshd) batches, the lighter SGLang memory fraction, and the
many-samples/high-temperature settings that surface reward variance — so the
recipe you write only sets the reward. It defaults to a `H100:2` single node;
pass `actor_num_gpus_per_node=8` (and a larger `num_rollout`) to use a full node.

To log training curves to W&B, also pass `wandb=WandbConfig(project="…")` to the
recipe — that needs a W&B account with write access, supplied via the
`wandb-secret` Modal secret.

`TrainConfig.train()` builds the Modal app, runs GRPO, and saves the trained
model as a Megatron checkpoint (exported to HuggingFace on demand at deploy).

In [ ]:
training_run = TrainConfig(
    model=Qwen3_ASR_1_7B(),
    dataset=train_dataset,
    recipe=Qwen3_ASR_1_7b_Recipe(custom_rm_function=word_error_rate_reward),
)
print("Starting training...")
train_result = training_run.train()
print(f"Training run id: {train_result.training_run_id}")

## Evaluate the trained checkpoint

`CustomDeployment.launch()` serves the trained checkpoint on SGLang
(converting the Megatron checkpoint to HuggingFace first, audio tower included).
Then we `POST` each clip to `/v1/audio/transcriptions`, scoring word
accuracy (`1 − WER`), and print the mean WER and mean accuracy.

In [ ]:
def transcribe_and_score(
    deployment: CustomDeployment, example: dict
) -> dict:
    import base64
    import io

    import jiwer
    import requests
    import soundfile as sf

    data_uri = example["audios"][0]
    reference = (example["label"] or "").lower().strip()
    b64 = data_uri.split(",", 1)[1] if data_uri.startswith("data:") else data_uri
    arr, sr = sf.read(io.BytesIO(base64.b64decode(b64)))

    buf = io.BytesIO()
    sf.write(buf, arr, sr, format="WAV")
    buf.seek(0)
    resp = requests.post(
        f"{deployment.url}/v1/audio/transcriptions",
        files={"file": ("clip.wav", buf, "audio/wav")},
        data={
            "model": deployment.served_model_name,
            "temperature": "0.0",
        },
        timeout=120,
    )
    resp.raise_for_status()
    hypothesis = (resp.json().get("text") or "").lower().strip()
    wer = float(jiwer.wer(reference, hypothesis)) if reference else 0.0

    return {
        "score": max(0.0, 1.0 - wer),
        "response": hypothesis,
        "wer": wer,
        "reference": reference,
    }

In [ ]:
checkpoint = list_checkpoints(train_result.training_run_id)[-1]
deployment = CustomDeployment.launch(
    Qwen3_ASR_1_7B(),
    checkpoint=checkpoint,
    unauthenticated=True,
)
print(f"Serving trained model at {deployment.url}")

def run_eval(
    deployment, *, max_concurrency: int = 2
) -> list[dict]:
    from concurrent.futures import ThreadPoolExecutor

    deployment.wait_until_ready(timeout=3000)

    def _score_one(example):
        return transcribe_and_score(deployment, example)

    with ThreadPoolExecutor(max_workers=max_concurrency) as executor:
        return list(executor.map(_score_one, eval_dataset.load()))

rows = run_eval(deployment)
mean_wer = sum(r["wer"] for r in rows) / len(rows) if rows else float("nan")
mean_acc = sum(r["score"] for r in rows) / len(rows) if rows else float("nan")
print(
    f"Eval: mean WER {mean_wer:.3f} "
    f"(accuracy {mean_acc:.3f}) over {len(rows)} clips"
)